# BhashaVerse/OneNMT adaptation · explicit language directions

A **separate experimental notebook** for pretrained BhashaVerse/OneNMT v3b in the repository's validated HF MBart-compatible path. It uses the checkpoint's **original SentencePiece and fairseq dictionary mapping**, not a generic HF AutoTokenizer. Language routing is expressed as `###src_lang-to-tgt_lang###` at preprocessing time; original JSONL source and target strings remain clean.

**Warning:** The HF model card's language list and the 36-language paper are not identical. Untouched-checkpoint generation must be validated against independent upstream outputs before making claims about new language adaptation. New directions may require the existing `repair-spm-extend → repair-append → repair-resize` pipeline, and sometimes full embedding training. See `docs/multilingual-adaptation.md` and `docs/repairs.md`.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# CPU corpus inspection is the default. Switch to train / train,repairs
# with a GPU and a fresh kernel before running heavyweight model cells.
SETUP_MODE = "preflight"  # base | preflight | train | train,repairs
REQUIRE_CUDA = SETUP_MODE.startswith("train")
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
ROOT = Path("/kaggle/working/BhaashikMTFinetuning" if Path("/kaggle/working").exists()
            else "/content/BhaashikMTFinetuning" if Path("/content").exists()
            else Path.cwd())
if not (ROOT / "pyproject.toml").is_file():
    subprocess.run(["git", "clone", "https://github.com/bhaashik/BhaashikMTFinetuning.git", str(ROOT)], check=True)
os.chdir(ROOT)
installer = ROOT / "scripts" / "setup_notebook.py"
assert installer.is_file(), "Refresh an outdated repository checkout before setup."
loaded = [name for name in ("torch", "transformers", "sentencepiece", "peft", "datasets")
          if name in sys.modules]
if loaded:
    raise RuntimeError("Restart the notebook kernel before installing new dependencies: "
                       + ", ".join(loaded))
command = [sys.executable, str(installer), "--extras", SETUP_MODE]
if REQUIRE_CUDA:
    command.append("--require-cuda")
subprocess.run(command, check=True)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from bhaashik.multilingual import require_routes, inspect_routes, write_tagged_copy
from bhaashik.pretrain_review import inspect_splits, plot_review, compare_dev
print("Repo:", ROOT, "Setup mode:", SETUP_MODE)
# The shared train,repairs setup installs HF MBart, SentencePiece and PEFT.
# Original BhashaVerse fairseq/SPM assets must be supplied separately.


## Immutable original assets, data and selected conditioning

A local HF MBart-format BhashaVerse checkpoint must contain `fairseq_dict.json`, `onemtv3b_spm.model`, `config.json` and complete safetensors. Use the original or a **new independently repaired candidate**, never mutate the original.

For unsupported language prefixes, this notebook checks whether the prefix can be represented by the existing dictionary and whether language directions collide. If not, stop and follow the separate tokenizer/dictionary/embedding repair steps. In that case `mode='full'` is required for new embeddings.


In [ ]:
from bhaashik.language_support import language_options
print("Focus-language coverage — native BhashaVerse support must still be validated:")
for item in language_options():
    print(item["name"], item["corpus_tag"], item["bhashaverse"])


In [ ]:
SRC, TGT = "mai_Deva", "hin_Deva"
PAIR = f"{SRC.split('_')[0]}-{TGT.split('_')[0]}"
DATA = ROOT / "data" / "local" / PAIR
ASSETS = ROOT / "data" / "local" / "onenmt-original"  # reviewed HF MBart assets
MODE = "lora"  # full is mandatory when new dictionary/embedding rows need training
STRATEGY = "text_prefix"
SEED = 42
RUN = ROOT / "runs" / f"bhashaverse-{PAIR}-{MODE}-s{SEED}"
REPORTS = ROOT / "runs" / f"bhashaverse-{PAIR}-review"
for split in ("train", "dev", "test"):
    assert (DATA / f"{split}.jsonl").is_file(), f"Supply {DATA / (split+'.jsonl')}"
    rows, routes = require_routes(DATA / f"{split}.jsonl")
    assert all("source_lang" in row and "target_lang" in row for row in rows)
    print(split, len(rows), routes)
CORPUS = inspect_splits({s: DATA / f"{s}.jsonl" for s in ("train","dev","test")}, REPORTS)
ROUTES = inspect_routes({s: DATA / f"{s}.jsonl" for s in ("train","dev","test")},
                        REPORTS / "language_routes.json")
for issue in CORPUS["issues"]:
    print(issue["severity"], issue["message"])
for fig in plot_review(REPORTS):
    print(fig)


## Model/corpus preflight

The source/target direction prefixes are tokenized using **FairseqCodec** and checked for unknown dictionary pieces and collisions. Run a separate *reference-parity* check with trusted original BhashaVerse inference before any research claims. If the prefix cannot be represented, do **not** automatically reuse another language's marker.

Existing repair commands are in `notebooks/02_onenmt_preflight_repairs.ipynb`. They create new candidate folders and keep old IDs stable. For `repair-resize` candidates, newly allocated weights must actually be trained.


In [ ]:
from bhaashik.bhashaverse_adaptation import validate_prefix_routes
from bhaashik.preflight import FairseqCodec, checkpoint_layout
CHECK_ASSETS = False
if CHECK_ASSETS:
    codec = FairseqCodec(ASSETS)
    layout = checkpoint_layout(codec)
    assert layout["weight_layout_checked"] and not layout["issues"], layout["issues"]
    all_rows = []
    from bhaashik.data import read_jsonl
    for split in ("train", "dev"):
        all_rows.extend(read_jsonl(DATA / f"{split}.jsonl"))
    print(validate_prefix_routes(codec, all_rows))
else:
    print("Asset preflight disabled; provide actual HF MBart-format weights before enabling.")


## Unfinetuned development baseline

A frozen development sample allows comparison of the unmodified local BhashaVerse checkpoint and adapted candidate. This code uses the project's own FairseqCodec-based regression generator; upstream parity must be separately checked. Source prefixes are inserted dynamically, never written into raw splits.


In [ ]:
from bhaashik.model_comparisons import select_dev_sample
from bhaashik.bhashaverse_adaptation import generate_bhashaverse_dev
from bhaashik.pretrain_review import compare_dev, plot_review
REPORTS.mkdir(parents=True, exist_ok=True)
DEV_SAMPLE = REPORTS / "sample.jsonl"
if not DEV_SAMPLE.exists():
    select_dev_sample(DATA / "dev.jsonl", DEV_SAMPLE, sample_size=100, seed=SEED)
RUN_BASELINE = False
DEV_PREDICTIONS = {}
if RUN_BASELINE:
    baseline = REPORTS / "bhashaverse-original-dev.jsonl"
    if not baseline.exists():
        generate_bhashaverse_dev(ASSETS, DEV_SAMPLE, baseline, batch_size=1)
    DEV_PREDICTIONS["BhashaVerse-original"] = baseline


## Optional supervised adaptation

The HF-only runner supports LoRA over unchanged embeddings, or full finetuning (including newly appended/initialized embedding rows from a prior validated repair). Every training/development row is labelled with explicit `source_lang` and `target_lang`. **No data are downloaded or altered automatically.**

Stop if independent checkpoint parity fails or if missing tokens/prefix collisions are detected. Model compatibility with standard HF MBart is a precondition, not a guarantee of parity with the upstream BhashaVerse implementation.


In [ ]:
from bhaashik.bhashaverse_adaptation import train_bhashaverse
APPROVE_TRAINING = False
if APPROVE_TRAINING:
    assert SETUP_MODE.startswith("train"), "Use train or train,repairs setup with a GPU."
    assert CORPUS["ready_for_training"], "Resolve the structural corpus errors first."
    assert CHECK_ASSETS, "Complete asset preflight first."
    config = {
        "model_assets": str(ASSETS), "output_dir": str(RUN),
        "train_file": str(DATA / "train.jsonl"),
        "dev_file": str(DATA / "dev.jsonl"),
        "strategy": "text_prefix", "mode": MODE, "epochs": 3,
        "seed": SEED, "batch_size": 1, "gradient_accumulation_steps": 16,
        "max_length": 128, "precision": "fp16",
    }
    cfg_path = ROOT / "configs" / f"bhashaverse-{PAIR}-{MODE}-s{SEED}.json"
    cfg_path.parent.mkdir(parents=True, exist_ok=True)
    cfg_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    print(train_bhashaverse(cfg_path))
else:
    print("Training disabled. Complete the independent model and corpus preflight.")


## Post-training development comparison and held-out test

Evaluate unchanged original and adapted checkpoint on identical dev IDs. Test-set evaluation must follow training approval/model selection and may need separate runs or more GPU RAM. Retain the fairseq dictionary and SPM with each exported checkpoint.


In [ ]:
from bhaashik.evaluation import evaluate, export_errors
RUN_POSTTRAIN = False
if RUN_POSTTRAIN:
    assert (RUN / "run_manifest.json").is_file(), "Attach or train adapted checkpoint."
    output = REPORTS / "bhashaverse-adapted-dev.jsonl"
    if not output.exists():
        generate_bhashaverse_dev(RUN, DEV_SAMPLE, output, batch_size=1)
    DEV_PREDICTIONS["BhashaVerse-adapted"] = output
    print(compare_dev(DEV_SAMPLE, DEV_PREDICTIONS, REPORTS))
    for figure in plot_review(REPORTS):
        print("Report figure:", figure)
    test_predictions = RUN / "predictions.jsonl"
    if not test_predictions.exists():
        generate_bhashaverse_dev(RUN, DATA / "test.jsonl", test_predictions)
    print(evaluate(test_predictions, RUN / "evaluation.json"))
    export_errors(test_predictions, RUN / "annotations.csv")
else:
    print("Post-training generation disabled.")


## Save selected artifacts

The fast ZIP includes the selected adapted model, its fairseq dictionary/SPM mapping, review reports and evaluation. `INCLUDE_CHECKPOINTS=True` preserves optimizer/Trainer states for resumption. Do not publish licensed/private data.


In [ ]:
from bhaashik.artifacts import archive_experiment
PRESERVE_ARTIFACTS = False
INCLUDE_CHECKPOINTS = False
if PRESERVE_ARTIFACTS:
    assert (RUN / "run_manifest.json").is_file()
    archive = archive_experiment(
        ROOT, RUN, ROOT.parent / f"bhashaverse-{PAIR}-{MODE}-s{SEED}.zip",
        report_dir=REPORTS, include_checkpoints=INCLUDE_CHECKPOINTS,
    )
    print("Preserve this archive:", archive)
else:
    print("Archiving disabled.")
